# FraudTrace AI - 01: Exploratory Data Analysis of Synthetic Forensic Evidence

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Explore evidence distributions, source types, entity types, and contradiction patterns across the forensic corpus.

| Attribute | Description |
| :--- | :--- |
| **Input** | ml/data/sample/ master tables and JSON collections |
| **Output** | Statistical summary tables and distribution plots |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Load Master Tables & Schema Overview
We load the relational master CSVs that define incidents, evidence, entities, matches, and ground-truth inconsistencies.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ml.src.data.loader import DataLoader

loader = DataLoader(dataset_type="sample")
master_data = loader.load_master_tables()

print("Master Tables Loaded:")
for name, df in master_data.items():
    print(f" - {name:<20}: {len(df):>4} records, columns: {list(df.columns)[:4]}...")


## 2. Evidence Type & Source Distribution
Understand what types of evidentiary sources are ingested (e.g. Bank Statements, FIRs, WhatsApp chats, CDRs).

In [ ]:
evidence_df = master_data.get('evidence', pd.DataFrame())
if not evidence_df.empty and 'evidence_type' in evidence_df.columns:
    plt.figure(figsize=(10, 4))
    type_counts = evidence_df['evidence_type'].value_counts()
    type_counts.plot(kind='bar', color='#3b82f6', edgecolor='black')
    plt.title('Evidence Type Distribution Across Corpus', fontsize=12, fontweight='bold')
    plt.xlabel('Evidence Type')
    plt.ylabel('Count')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    print("Evidence Counts by Type:")
    print(type_counts)
else:
    print("Evidence table not found or empty.")


## 3. Inconsistency & Contradiction Distribution
Analyze the prevalence of ground-truth inconsistencies (amount mismatches, timestamp drifts, entity discrepancies).

In [ ]:
inconsistencies_df = master_data.get('inconsistencies', pd.DataFrame())
if not inconsistencies_df.empty:
    cat_col = 'inconsistency_type' if 'inconsistency_type' in inconsistencies_df.columns else inconsistencies_df.columns[1]
    plt.figure(figsize=(8, 4))
    inconsistencies_df[cat_col].value_counts().plot(kind='barh', color='#ef4444', edgecolor='black')
    plt.title('Ground-Truth Inconsistency Distribution', fontsize=12, fontweight='bold')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.show()
    print(inconsistencies_df[cat_col].value_counts())
else:
    print("Inconsistencies table empty or unavailable.")


## 4. Key Takeaways
- Evidence spans diverse modalities: Banking core logs, police FIRs, SMS alerts, telecom CDRs.
- Contradictions occur naturally due to human reporting error, bank processing fees, and unsynchronized clocks.
- Machine learning and deterministic rules must work together to isolate discrepancies without bias.